# Notebook 01 — Data Loading & QA Dataset (Kaggle T4)

In [11]:
!pip install -q groq tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00a 0:00:01


In [1]:
import os; print(os.listdir('/kaggle/input'))

['datasets']


In [4]:
import os, sys, subprocess, glob

WORKING_DIR = '/kaggle/working'

# Search entire /kaggle/input for indiana_reports.csv
csv_matches = glob.glob('/kaggle/input/**/indiana_reports.csv', recursive=True)
if not csv_matches:
    direct = '/kaggle/input/chest-xrays-indiana-university/indiana_reports.csv'
    if os.path.exists(direct): csv_matches = [direct]

if not csv_matches:
    print('Contents of /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            print(f'  DIR: {os.path.join(root, d)}')
        for f in files[:5]:
            print(f'  FILE: {os.path.join(root, f)}')
        if root.count(os.sep) > 5:
            break
    raise RuntimeError('indiana_reports.csv not found - add Kaggle Input dataset')

INPUT_DIR = os.path.dirname(csv_matches[0])
print(f'✓ Dataset found at {INPUT_DIR}')
print(f'\nContents:')
for f in os.listdir(INPUT_DIR):
    print(f'  {f}')

✓ Dataset found at /kaggle/input/datasets/raddar/chest-xrays-indiana-university

Contents:
  indiana_projections.csv
  images
  indiana_reports.csv


In [5]:
# Get API keys from Kaggle secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret('GROQ_API_KEY')
HF_TOKEN = user_secrets.get_secret('HF_TOKEN')

print('✓ Secrets loaded')

✓ Secrets loaded


In [6]:
# Clone repo
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')
if not os.path.exists(REPO_PATH):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    print('Updating repository...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)
print('✓ Repository ready')

Cloning repository...
✓ Repository ready


In [7]:
# Auto-detect image directory in Kaggle input
import glob

png_files = glob.glob(os.path.join(INPUT_DIR, '**', '*.png'), recursive=True)
if not png_files:
    raise RuntimeError(f'No PNG images found in {INPUT_DIR}')

IMAGES_DIR = os.path.dirname(png_files[0])
print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')

Found 7470 PNG images in: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized


In [8]:
# Load dataset from Kaggle CSVs (in /kaggle/input/)
from src.data.openi_loader import OpenILoader

loader = OpenILoader(images_dir=IMAGES_DIR)
df = loader.load_from_kaggle_csvs(kaggle_dir=INPUT_DIR)

print(f'Loaded {len(df)} studies')
print(df.head(3))

Loaded 3652 studies
   study_id                                         impression  \
0         1                               Normal chest x-XXXX.   
1         2                       No acute pulmonary findings.   
2         3  No displaced rib fractures, pneumothorax, or p...   

                                            findings  \
0  The cardiac silhouette and mediastinum size ar...   
1  Borderline cardiomegaly. Midline sternotomy XX...   
2                                                      

                                          image_path  
0  /kaggle/input/datasets/raddar/chest-xrays-indi...  
1  /kaggle/input/datasets/raddar/chest-xrays-indi...  
2  /kaggle/input/datasets/raddar/chest-xrays-indi...  


In [9]:
# Train/val/test split
import pandas as pd

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

corpus_path = os.path.join(WORKING_DIR, 'reports_corpus.csv')
full_df.to_csv(corpus_path, index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Saved: {corpus_path}')

Train: 2921 | Val: 365 | Test: 366
Saved: /kaggle/working/reports_corpus.csv


In [12]:
# QA Generation via Groq
from src.data.qa_creator import QACreator

creator = QACreator(groq_api_key=GROQ_API_KEY)
QA_OUTPUT = os.path.join(WORKING_DIR, 'qa_dataset.jsonl')

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=200,  # ~3,000 pairs
)

print(f'Generated {len(pairs)} QA pairs')

Generating QA pairs: 100%|██████████| 200/200 [59:36<00:00, 17.88s/it] 

Generated 1515 QA pairs → /kaggle/working/qa_dataset.jsonl
Generated 1515 QA pairs


In [13]:
# Stats and samples
import json

qa_df = pd.read_json(QA_OUTPUT, lines=True)
print(f'Total pairs   : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'\nSplit:\n{qa_df["split"].value_counts()}')
print(f'\nCategory:\n{qa_df["category"].value_counts()}')

print('\n=== Sample QA pairs ===')
for s in qa_df.head(3).to_dict('records'):
    print(f"Q: {s['question']}")
    print(f"A: {s['answer']}\n")

print(f'\n✓ All outputs in {WORKING_DIR}')

Total pairs   : 1515
Unique studies: 200

Split:
split
train    1515
Name: count, dtype: int64

Category:
category
No Finding              446
Consolidation           281
Pleural Effusion        189
Lung Lesion             118
Lung Opacity            109
Enlarged Mediastinum     89
Atelectasis              78
Cardiomegaly             54
Support Devices          45
Pneumothorax             39
Edema                    37
Fracture                 16
Pneumonia                12
Pleural Other             2
Name: count, dtype: int64

=== Sample QA pairs ===
Q: Is there consolidation visible in this chest X-ray?
A: No consolidation is observed in the lungs.

Q: Can pulmonary consolidation be identified?
A: No, pulmonary consolidation is not observed. The lungs are free of focal airspace disease.

Q: Are there signs of airspace consolidation?
A: No, airspace consolidation is not observed. The lungs are free of focal airspace disease.


✓ All outputs in /kaggle/working
